1º Desafio: Dado uma lista de postings, gerar a lista de d-gaps.

DocIds = [85, 93, 97, 240, 2010]

gaps = [85, 8, 4, 143, 1770]

In [ ]:
import array
from typing import List

def docids_to_dgaps(docids: List[int]) -> List[int]:
  """Converter DocIDs ordenados em d-gaps (deltas)."""
  if not docids:
    return []
  gaps = [docids[0]]
  for i in range(1, len(docids)):
    gaps.append(docids[i] - docids[i-1])
  return gaps

def dgaps_to_docids(dgaps: List[int]) -> List[int]:
  """Converter d-gaps em DocIDs ordenados."""
  if not dgaps:
    return []
  docids = [dgaps[0]]
  accumulator = dgaps[0]
  for gap in dgaps[1:]:
    accumulator += gap
    docids.append(accumulator)
  return docids

def vb_encode(numbers: List[int]) -> bytes:
  """Codificar uma lista de números inteiros para o formato VB.

  Convenção: MSB = 1 marca o último byte do número.
  """
  stream = bytearray()
  for n in numbers:
    buffer = []
    while True:
      buffer.append(n % 128)
      if n < 128:
        break
      n //= 128

    # Inverter para manter a ordem dos bytes mais significativos primeiro
    buffer.reverse()
    # Marca o stop bit (MSB = 1) no ultimo byte
    buffer[-1] |= 0x80
    # Adicionar o buffer ao stream
    stream.extend(buffer)
  return bytes(stream)

def vb_decode(stream: bytes) -> List[int]:
  """Decodificar uma lista de números inteiros do formato VB."""
  numbers = []
  n = 0
  for byte in stream:
    if byte & 0x80 == 0:
      # MSB == 0: byte intermediario, acumula 7 bits e continuar
      n = n * 128 + byte
    else:
      # MSB == 1: byte final, extrai os 7 bits do payload e emite o número
      n = n * 128 + (byte & 0x7F)
      numbers.append(n)
      n = 0
  return numbers

### --- Pipeline integrado para conveniência ---

def compress_postings(doc_ids: List[int]) -> bytes:
  """Pipeline completo: DocIDs --> d-gaps --> VB Code bytes."""
  gaps = docids_to_dgaps(doc_ids)
  return vb_encode(gaps)

def decompress_postings(vb_bytes: bytes) -> List[int]:
  """Pipeline completo: VB Code bytes --> d-gaps --> DocIDs."""
  gaps = vb_decode(vb_bytes)
  return dgaps_to_docids(gaps)

In [ ]:
docids = list([85, 93, 97, 240, 2010])
print(docids)

posting_compressed = compress_postings(docids)
print(posting_compressed)

posting_decompressed = decompress_postings(posting_compressed)
print(posting_decompressed)


[85, 93, 97, 240, 2010]
b'\xd5\x88\x84\x01\x8f\r\xea'
[85, 93, 97, 240, 2010]


In [ ]:
import random
import time

def run_bechmark():
  print("Gerando 200.000 DocIDs ordenados...")
  random.seed(42)
  doc_count = 200_000
  universe_size = 5_000_000
  doc_ids = sorted(random.sample(range(1, universe_size + 1), doc_count))

  raw_array = array.array('I', doc_ids)
  raw_bytes_size = len(raw_array) * raw_array.itemsize

  print(f"Total de documentos indexados: {doc_count}")
  print(f"Tamanho do array de DocIDs (bytes): {raw_bytes_size / 1024: .2f} KB \n")

  ## Com compressão

  # Compressão
  start_time = time.perf_counter()
  posting_compressed = compress_postings(doc_ids)
  encode_duration = time.perf_counter() - start_time

  compressed_size = len(posting_compressed)
  savings = (1 - (compressed_size / raw_bytes_size)) * 100

  print("--- RESULTADOS DE COMPRESSÃO ---")
  print(f"Tamanho comprimido (VB Code):   {compressed_size / 1024:.2f} KB")
  print(f"Redução de espaço:              {savings:.2f}%")
  print(f"Bytes médios por CodID:         {compressed_size / doc_count:.2f} bytes")
  print(f"Tempo de codificação:           {encode_duration * 1000:.2f} ms")
  print(f"Throughtput de codificação:     {doc_count / encode_duration:.0f} docs/s \n")

  # Descompressão
  start_time = time.perf_counter()
  recovered_doc_ids = decompress_postings(posting_compressed)
  decode_duration = time.perf_counter() - start_time

  print("--- RESULTADOS DE DESCOMPRESSÂO ---")
  print(f"Tempo de decodificação:         {decode_duration * 1000:.2f} ms")
  print(f"Throughtput de decodificação:   {doc_count / decode_duration:.0f} docs/s \n")

  # Verificação de Integridade
  assert recovered_doc_ids == doc_ids, "Erro: Os DocIDs decodificados não coincidem com os originais!"
  print("Integridade verificada com sucesso!")

run_bechmark()

Gerando 200.000 DocIDs ordenados...
Total de documentos indexados: 200000
Tamanho do array de DocIDs (bytes):  781.25 KB 

--- RESULTADOS DE COMPRESSÃO ---
Tamanho comprimido (VB Code):   196.37 KB
Redução de espaço:              74.86%
Bytes médios por CodID:         1.01 bytes
Tempo de codificação:           207.91 ms
Throughtput de codificação:     961940 docs/s 

--- RESULTADOS DE DESCOMPRESSÂO ---
Tempo de decodificação:         70.46 ms
Throughtput de decodificação:   2838394 docs/s 

Integridade verificada com sucesso!
